# Quinta Playa — M3C2 beach change starter (py4dgeo)

Starting point for comparing two months of Quinta Playa point clouds with `py4dgeo`.
Adapted from py4dgeo's own tutorials (`m3c2.ipynb`, `registration.ipynb`) and the
applied rock-glacier example (`demo/m3c2-change_analysis.ipynb`), pointed at our
own data instead of the toy/synthetic datasets those use.

**Assumes**, per `claude_dem-completion-plan.md`:
- Both months' point clouds are already GCP-corrected in Metashape (right GCP file
  chosen per that month's A/B test) and exported as the **dense cloud**, not the mesh.
- You've exported dense point clouds (not just the DEM raster) for at least two months
  — e.g. Dec 2024 and Jan 2025, the first two with clean GCP fits.

**Not yet handled here:** the water-body masking / noise filtering from the DEM
pipeline — do that before exporting the point cloud used below, or filter here.


In [1]:
import py4dgeo
import numpy as np
import matplotlib.pyplot as plt

print(py4dgeo.__version__ if hasattr(py4dgeo, "__version__") else "py4dgeo loaded")


1.2.0


## 1. Load the two epochs

Update these paths to wherever you exported each month's dense point cloud
(LAZ/LAS from Metashape: Export → Dense Cloud). These are **not** the DEM
GeoTIFFs used in the QGIS boundary pipeline — M3C2 needs the actual 3D point
cloud, not a rasterized surface.


In [2]:
# --- EDIT THESE PATHS ---
epoch1_path = "/Users/giovannilivibacci/Documents/MDP Galapagos/Drone/Finalised/2025/December/PointCloud_Dec_2025.laz"   # reference epoch (earlier month)
epoch2_path = "/Users/giovannilivibacci/Documents/MDP Galapagos/Drone/Finalised/2025/January/PointCloud_Jan_2025.laz"    # comparison epoch (later month)

epoch1, epoch2 = py4dgeo.read_from_las(
    epoch1_path,
    epoch2_path,
    # additional_dimensions={"point_source_id": "scanpos_id"},  # if you carry per-flight source IDs
)

print(epoch1.cloud.shape, epoch2.cloud.shape)


[2026-09-14 10:34:40][INFO] Reading point cloud from file '/Users/giovannilivibacci/Documents/MDP Galapagos/Drone/Finalised/2025/December/PointCloud_Dec_2025.laz'
[2026-09-14 10:35:01][INFO] Reading point cloud from file '/Users/giovannilivibacci/Documents/MDP Galapagos/Drone/Finalised/2025/January/PointCloud_Jan_2025.laz'


: 

## 2. Registration check on stable ground

Per the plan doc: even after each month's own GCP A/B test converges to a good
fit, there's no guarantee two months land on *exactly* the same absolute frame.
Before running M3C2 on the whole beach, do a fine ICP registration restricted to
a stable, non-eroding feature (rock outcrop, building corner, fixed structure)
to check/correct for residual offset.

Isolate a stable subset of both clouds (by bounding box or a saved polygon) and
compare it here. See `demo_workflows/registration_standard_ICP.ipynb` for the
fuller worked pattern this is based on.


In [ ]:
# --- EDIT: bounding box (in your export CRS, e.g. UTM 15S / EPSG:32715)
# around a stable, non-eroding feature visible in both epochs ---
stable_bbox = None  # e.g. ((xmin, ymin), (xmax, ymax))

# Sketch only — see registration_standard_ICP.ipynb for the real API calls
# (py4dgeo.iterative_closest_point or similar registration routine) and adapt
# once you've picked your stable feature.
if stable_bbox is not None:
    pass  # TODO: subset both epochs to stable_bbox, run ICP, inspect the
          # resulting transformation before deciding whether/how to apply it
          # to the full epoch2 cloud.


## 3. Core points + M3C2

Core points are where distances get computed — commonly a subsampled version
of the reference epoch. Start coarse (e.g. every Nth point or a fixed spacing)
and tighten once the run time is known; beach point clouds from a few thousand
photos can be large.

`normal_radii` and the M3C2 cylinder radius are the two parameters most worth
tuning for beach sand vs. the rock-glacier example's terrain — start from the
tutorial's defaults and check `m3c2.ipynb` / `m3c2-change_analysis.ipynb` for
how they picked theirs, then sanity-check against known beach roughness scale.


In [ ]:
corepoints = epoch1.cloud[::50]  # crude subsample as a starting point — revisit

m3c2 = py4dgeo.M3C2(
    epochs=(epoch1, epoch2),
    corepoints=corepoints,
    normal_radii=(1.0,),   # TODO: tune for beach sand roughness
    cyl_radius=1.0,        # TODO: tune
)

distances, uncertainties = m3c2.run()


## 4. Look at the result

Quick sanity plot before trusting anything. Positive/negative sign convention
depends on epoch order — confirm which direction means accretion vs. erosion
for your setup before reading too much into it.


In [ ]:
finite = np.isfinite(distances)
plt.figure(figsize=(8, 5))
sc = plt.scatter(
    corepoints[finite, 0], corepoints[finite, 1],
    c=distances[finite], cmap="RdBu", vmin=-1, vmax=1, s=2
)
plt.colorbar(sc, label="M3C2 distance (m)")
plt.gca().set_aspect("equal")
plt.title("Dec 2024 -> Jan 2025 (placeholder — confirm epoch order)")
plt.show()


## Next steps

- Once a third+ month is processed with its own GCP A/B test, revisit this as a
  **4D Objects-by-Change** analysis instead of a single pairwise M3C2
  (`official_tutorials/4dobc-creation.ipynb` → `4dobc-analysis.ipynb`,
  applied version in `demo_workflows/4dobc-change_analysis.ipynb`) to track
  change across the whole time series rather than one before/after pair.
- If registration/GCP fit quality keeps varying month to month, `M3C2-EP`
  (`official_tutorials/m3c2ep.ipynb`, `demo_workflows/m3c2ep_change_analysis.ipynb`)
  lets that uncertainty propagate into the distance result instead of treating
  every month as equally trustworthy.
- Cross-check this notebook's transect/profile-based output against the
  existing geodesic profile script (`MatanYuval/WorkWithMaca`) as a second
  opinion, per the protocol review in the plan doc.
